# TFP-BD Final Evaluation (70/20/10 split, all detectors)

Ei notebook EKBAR chalale 5 ta detector (YOLOv8n, YOLO11n, RT-DETR-L, Faster R-CNN, EfficientDet-D3) er jonno
**same test set, same frame, same niyom** e ber hobe:

1. **Detection:** test set e mAP50, mAP50-95, per-class AP50 (sob model e ekই torchmetrics code)
2. **Roadside Encroachment:** camera-wise polygon zone + homography alignment, validation e threshold bachai, test e ekbar
3. **Traffic Congestion:** threshold train set er annotation count theke (33rd/66th percentile), model er confidence validation e bachai, test e ekbar
4. **McNemar test** (model gulor encroachment parthokko statistically significant kina) + **master table**

Sob result `/kaggle/working/eval_results/` e CSV hisebe save hobe, sheshe `eval_results.zip`.

**Settings:** Accelerator = GPU T4 x2 (ba P100), Internet = ON. Prothome Cell 1-2 run kore check koro sob input OK; tarpor **Save Version -> Save & Run All (Commit)**.

In [ ]:
# CELL 1: library install (2-3 minute)
!pip install -q ultralytics effdet torchmetrics pycocotools
import ultralytics, torchmetrics
print("ultralytics", ultralytics.__version__, "| torchmetrics", torchmetrics.__version__)

In [ ]:
# CELL 2: config + sob input file khuje ber kora
import os, re, glob, time, math, pickle, shutil, gc, warnings
import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from matplotlib.path import Path
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
warnings.filterwarnings("ignore")

WK   = "/kaggle/working"
OUTD = f"{WK}/eval_results"
os.makedirs(OUTD, exist_ok=True)
DEV  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEV, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(GPU nai!)")

CLASSES  = ["Car", "Bus", "Truck", "Motorcycle", "Bicycle", "Rickshaw", "Mini-Truck", "CNG", "Pedestrian"]
PED      = {8}
LOC_NAME = {1: "Arambag", 2: "Shapla Chattar", 3: "Abul Hotel", 4: "Bashabo"}
EXTS     = (".jpg", ".jpeg", ".png", ".bmp")

# kon kon model chalabo (kono ekta baad dite chaile list theke muche dao)
MODELS_TO_RUN = ["YOLOv8n", "YOLO11n", "RT-DETR-L", "Faster R-CNN", "EfficientDet-D3"]
THRS = [round(x, 2) for x in np.arange(0.10, 0.61, 0.05)]   # confidence sweep (validation)
EFF_IMG, EFF_SHIFT = 640, -1     # EfficientDet: input size, class offset (ager run e confirm kora)
RUN_ULTRA_CHECK = True           # YOLO/RT-DETR er Ultralytics official mAP o ber korbo (cross-check)

BIG_DIRS = {"tfp_dataset", "Bangladeshi Traffic Flow Dataset", "images", "labels"}
F = {"ORIG": None, "SH": None, "ALIGN": None, "YOLOv8n": None, "YOLO11n": None,
     "RT-DETR-L": None, "Faster R-CNN": None, "EfficientDet-D3": None}
eff_files, eff_dirs = [], []
for root, dirs, files in os.walk("/kaggle/input"):
    low = root.lower()
    if F["ORIG"] is None and "Bangladeshi Traffic Flow Dataset" in dirs:
        c = os.path.join(root, "Bangladeshi Traffic Flow Dataset")
        if os.path.isdir(os.path.join(c, "Annotated Images")):
            F["ORIG"] = c
    if F["SH"] is None and "tfp_dataset" in dirs:
        c = os.path.join(root, "tfp_dataset")
        if all(os.path.isdir(os.path.join(c, s, "images")) for s in ("train", "val", "test")):
            F["SH"] = c
    dirs[:] = [d for d in dirs if d not in BIG_DIRS]
    for f in files:
        p = os.path.join(root, f)
        if f == "sharmin_yolov8n_7030_best.pt":
            F["YOLOv8n"] = p
        elif f == "best.pt" and "yolov11" in low:
            F["YOLO11n"] = p
        elif f == "best.pt" and "rtdetr" in low:
            F["RT-DETR-L"] = p
        elif f == "faster_rcnn_epoch_10_FINAL.pth":
            F["Faster R-CNN"] = p
        elif f == "alignment_cache.pkl":
            F["ALIGN"] = p
        elif ("effdet" in low or "efficientd3" in low) and f.lower().endswith((".zip", ".pth", ".pt")):
            eff_files.append(p)
    if ("effdet" in low or "efficientd3" in low) and "data.pkl" in files:
        eff_dirs.append(root)

if eff_files:
    F["EfficientDet-D3"] = eff_files[0]
elif eff_dirs:
    # Kaggle Model zip ke folder kore rakhe; torch.load er jonno abar zip banai
    d = eff_dirs[0]
    F["EfficientDet-D3"] = shutil.make_archive(os.path.join(WK, os.path.basename(d)), "zip",
                                               root_dir=os.path.dirname(d), base_dir=os.path.basename(d))

ORIG, SH = F["ORIG"], F["SH"]
print("\n" + "=" * 70)
for k, v in F.items():
    need = k not in ("ALIGN",) and (k in ("ORIG", "SH") or k in MODELS_TO_RUN)
    tag = "OK " if v else ("-- (optional)" if not need else "MISSING")
    print(f"{tag:14s} {k:16s} {v if v else ''}")
print("=" * 70)
missing = [k for k, v in F.items() if v is None and (k in ("ORIG", "SH") or k in MODELS_TO_RUN)]
assert not missing, f"Ei input gulo pawa jayni, Add Input e jog koro: {missing}"
print("Sob input pawa geche. (ALIGN '--' thakle somossa nai: alignment ekhane notun kore hobe ~30 min)")

In [ ]:
# CELL 3: camera-wise footpath zone + homography alignment (ekbar hisab kore cache)
ZONES = {
    (1, "DL", 1): [[(0, .45), (.24, .45), (.235, .55), (.165, .74), (.065, 1), (0, 1)],
                   [(.68, .56), (.8, .61), (1, .74), (1, .92), (.85, .79), (.75, .72), (.66, .63)]],
    (1, "DL", 2): [[(0, .45), (.27, .45), (.25, .55), (.195, .74), (.105, 1), (0, 1)],
                   [(.68, .57), (.78, .61), (.95, .69), (.95, .92), (.85, .84), (.7, .75), (.66, .68)]],
    (1, "DL", 3): [[(0, 0), (.25, 0), (.24, .12), (.2, .24), (.12, .36), (.03, .45), (0, .47)],
                   [(.72, .1), (.85, .1), (1, .3), (1, .65), (.9, .5), (.78, .32), (.72, .2)]],
    (1, "DL", 4): [[(0, .45), (.24, .45), (.235, .55), (.165, .74), (.065, 1), (0, 1)],
                   [(.68, .53), (.8, .57), (1, .68), (1, .88), (.85, .75), (.75, .68), (.66, .6)]],
    (1, "DL", 5): [[(0, .15), (.12, .15), (.12, .3), (.08, .45), (.03, .6), (0, .65)],
                   [(.6, .28), (1, .28), (1, .42), (.62, .38)]],
    (1, "SL", 1): [[(0, .5), (.2, .44), (.33, .42), (.32, .52), (.29, .62), (.16, 1), (0, 1)]],
    (1, "SL", 2): [[(0, .5), (.2, .42), (.35, .4), (.34, .48), (.3, .6), (.12, 1), (0, 1)]],
    (1, "SL", 3): [[(0, .45), (.2, .38), (.35, .36), (.34, .48), (.3, .6), (.12, 1), (0, 1)]],
    (1, "SL", 4): [[(0, .5), (.2, .42), (.35, .4), (.34, .48), (.3, .6), (.12, 1), (0, 1)]],
    (1, "SL", 5): [[(0, .3), (.3, .3), (.32, .42), (.27, .6), (.2, .8), (.15, 1), (0, 1)]],
    (3, "DL", 1): [[(.55, .05), (.68, 0), (1, 0), (1, .5), (.76, .5), (.68, .4), (.6, .2)],
                   [(0, .05), (.2, .05), (.2, .2), (.12, .35), (0, .42)]],
    (3, "DL", 2): [[(.64, 0), (1, 0), (1, .55), (.9, .42), (.78, .27), (.7, .15)],
                   [(0, 0), (.2, 0), (.15, .2), (.1, .4), (.06, .6), (0, .7)]],
    (3, "DL", 3): [[(.6, 0), (1, 0), (1, .7), (.92, .55), (.8, .35), (.7, .2)],
                   [(0, 0), (.2, 0), (.13, .2), (.07, .4), (.03, .55), (0, .6)]],
    (3, "DL", 4): [[(.6, 0), (1, 0), (1, .45), (.85, .35), (.72, .2), (.62, .08)],
                   [(0, .05), (.22, .05), (.18, .25), (.12, .45), (.08, .6), (0, .65)]],
    (3, "DL", 5): [[(.68, 0), (1, 0), (1, 1), (.92, 1), (.8, .6), (.72, .3)],
                   [(0, .1), (.2, .1), (.15, .3), (.1, .5), (.05, .65), (0, .7)]],
    (3, "SL", 1): [[(.52, .2), (.62, .15), (1, .45), (1, 1), (.78, 1), (.68, .75), (.6, .55), (.54, .35)]],
    (3, "SL", 2): [[(.58, 0), (1, 0), (1, .72), (.9, .6), (.75, .35), (.62, .1)]],
    (3, "SL", 3): [[(.62, 0), (1, 0), (1, 1), (.92, 1), (.85, .6), (.75, .4), (.66, .2)]],
    (3, "SL", 4): [[(.62, 0), (1, 0), (1, .55), (.9, .4), (.75, .25), (.64, .12)]],
    (3, "SL", 5): [[(.58, 0), (1, 0), (1, .6), (.92, .62), (.8, .4), (.66, .2)]],
    (4, "DL", 1): [[(.55, .14), (.64, .24), (.8, .4), (.9, .49), (1, .56), (1, .7), (.9, .62), (.8, .54), (.64, .38), (.56, .22)],
                   [(0, .15), (.22, .15), (.2, .3), (.12, .45), (.05, .55), (0, .6)]],
    (4, "DL", 2): [[(.6, .4), (.7, .44), (.85, .5), (1, .56), (1, .72), (.85, .63), (.7, .55), (.6, .48)],
                   [(0, .4), (.2, .4), (.22, .5), (.18, .62), (.12, .75), (0, .85)]],
    (4, "DL", 3): [[(.55, .18), (.7, .32), (.85, .46), (1, .56), (1, .78), (.85, .64), (.75, .56), (.65, .5), (.58, .38), (.53, .25)],
                   [(0, .2), (.2, .2), (.25, .3), (.2, .45), (.12, .6), (.05, .75), (0, .8)]],
    (4, "DL", 4): [[(.58, .18), (.75, .25), (.9, .33), (1, .4), (1, .62), (.88, .52), (.75, .45), (.62, .35), (.56, .25)],
                   [(0, .15), (.22, .15), (.25, .25), (.2, .4), (.12, .5), (0, .55)]],
    (4, "DL", 5): [[(.58, .22), (.75, .32), (.9, .42), (1, .47), (1, .65), (.88, .58), (.75, .5), (.6, .35)],
                   [(0, .25), (.25, .25), (.28, .35), (.22, .55), (.15, .75), (.1, 1), (0, 1)]],
    (4, "SL", 1): [[(.48, .33), (.6, .42), (.75, .58), (.9, .72), (1, .78), (1, .92), (.88, .82), (.72, .68), (.58, .52), (.46, .4)]],
    (4, "SL", 2): [[(.48, .4), (.62, .5), (.75, .6), (.9, .72), (1, .78), (1, .95), (.88, .86), (.72, .74), (.6, .62), (.47, .48)]],
    (4, "SL", 3): [[(.5, .22), (.62, .28), (.72, .36), (.86, .52), (.95, .64), (1, .7), (1, .82), (.92, .76), (.84, .66), (.72, .54), (.6, .4), (.49, .3)]],
    (4, "SL", 4): [[(.53, .33), (.7, .45), (.85, .56), (1, .66), (1, 1), (.85, .9), (.75, .8), (.6, .6), (.52, .42)]],
    (4, "SL", 5): [[(0, .4), (.25, .4), (.35, .45), (.33, .6), (.3, .8), (.28, 1), (0, 1)]],
}

def key_of(p):
    m = re.search(r"Location (\d)", p)
    t = re.match(r"(\d)-LOC", os.path.basename(os.path.dirname(p)))
    lane = "DL" if "Double" in p else ("SL" if "Single" in p else None)
    return (int(m.group(1)), lane, int(t.group(1))) if (m and t and lane) else None

def stem_of(p):
    return os.path.splitext(os.path.basename(p))[0]

key_of_stem, folder_frames = {}, {}
for p in glob.glob(f"{ORIG}/Annotated Images/**/*", recursive=True):
    if p.lower().endswith(EXTS):
        k = key_of(p)
        key_of_stem[stem_of(p)] = k
        if k in ZONES:
            folder_frames.setdefault(k, []).append(p)
print(f"Camera folder: {len(folder_frames)} / 30")

W0, H0 = 640, 480
CORN = np.float32([[0, 0], [W0, 0], [W0, H0], [0, H0]]).reshape(-1, 1, 2)
MIN_INL, MIN_SCORE = 30, 0.15

def load(p):
    img = cv2.imread(p)
    if img is None:
        return None
    g = cv2.cvtColor(cv2.resize(img, (W0, H0)), cv2.COLOR_BGR2GRAY)
    gs = cv2.createCLAHE(2.0, (8, 8)).apply(cv2.resize(g, (480, 360)))
    kp, d = cv2.SIFT_create(2000).detectAndCompute(gs, None)
    if d is None or len(kp) < 20:
        return None
    return {"g": g, "pts": np.float32([k.pt for k in kp]) * (W0 / 480), "d": d}

def sane(H):
    q = cv2.perspectiveTransform(CORN, H)
    area = cv2.contourArea(q) / (W0 * H0)
    return 0.3 < area < 3.0 and cv2.isContourConvex(q.astype(np.float32))

def homog(fa, fb):
    if fa is None or fb is None:
        return None, 0
    kn = cv2.FlannBasedMatcher(dict(algorithm=1, trees=5), dict(checks=32)).knnMatch(fa["d"], fb["d"], k=2)
    good = [x[0] for x in kn if len(x) == 2 and x[0].distance < 0.75 * x[1].distance]
    if len(good) < MIN_INL:
        return None, len(good)
    A = fa["pts"][[x.queryIdx for x in good]]
    B = fb["pts"][[x.trainIdx for x in good]]
    H, inl = cv2.findHomography(A, B, cv2.USAC_MAGSAC, 3.0, maxIters=5000, confidence=0.999)
    if H is None or inl is None:
        return None, 0
    n = int(inl.sum())
    if n < MIN_INL or not sane(H):
        return None, n
    return H / H[2, 2], n

def _grad(g):
    g = cv2.GaussianBlur(g.astype(np.float32), (0, 0), 1.5)
    return cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1)

def score(fref, f, H):
    wa = cv2.warpPerspective(fref["g"], H, (W0, H0))
    wm = cv2.warpPerspective(np.full((H0, W0), 255, np.uint8), H, (W0, H0), flags=cv2.INTER_NEAREST)
    valid = cv2.erode(wm, np.ones((9, 9), np.uint8)) > 0
    if valid.mean() < 0.15:
        return 0.0
    ax, ay = _grad(wa)
    bx, by = _grad(f["g"])
    num = (ax * bx + ay * by)[valid].sum()
    den = np.sqrt((ax ** 2 + ay ** 2)[valid].sum() * (bx ** 2 + by ** 2)[valid].sum()) + 1e-6
    return float(num / den)

def zshift(k, H):
    d = []
    for poly in ZONES[k]:
        pts = np.float32([[x * W0, y * H0] for x, y in poly]).reshape(-1, 1, 2)
        q = cv2.perspectiveTransform(pts, H).reshape(-1, 2) - pts.reshape(-1, 2)
        d += list(np.hypot(q[:, 0] / W0, q[:, 1] / H0))
    return float(np.median(d) * 100)

def fnum(p):
    m = re.search(r"frame(\d+)", os.path.basename(p))
    return int(m.group(1)) if m else 0

def align_folder(k):
    fl = sorted(folder_frames[k])
    ref = fl[0]
    fr = load(ref)
    out = {stem_of(ref): (k, np.eye(3), "ref", 0.0, 1.0)}
    last = (fr, np.eye(3))
    for p in sorted(fl[1:], key=fnum):
        f = load(p)
        best = (None, "fail", 0.0)
        H, _ = homog(fr, f)
        if H is not None:
            s = score(fr, f, H)
            best = (H, "direct", s) if s >= MIN_SCORE else (None, "fail", s)
        if best[0] is None and last[0] is not None and last[0] is not fr:
            Hc, _ = homog(last[0], f)
            if Hc is not None:
                H2 = Hc @ last[1]
                H2 = H2 / H2[2, 2]
                if sane(H2):
                    s2 = score(fr, f, H2)
                    if s2 >= MIN_SCORE:
                        best = (H2, "chain", s2)
        H, how, s = best
        if H is None:
            out[stem_of(p)] = (k, None, "fail", np.nan, s)
        else:
            out[stem_of(p)] = (k, H, how, zshift(k, H), s)
            last = (f, H)
    return k, out


if F["ALIGN"]:
    AL = pickle.load(open(F["ALIGN"], "rb"))
    print(f"Alignment cache load holo: {F['ALIGN']} ({len(AL)} frame)")
else:
    t0 = time.time()
    AL = {}
    with ThreadPoolExecutor(max_workers=4) as ex:
        for i, (k, out) in enumerate(ex.map(align_folder, sorted(folder_frames))):
            AL.update(out)
            if (i + 1) % 5 == 0:
                print(f"  alignment: {i + 1}/{len(folder_frames)} folder ({time.time() - t0:.0f}s)")
    pickle.dump(AL, open(f"{WK}/alignment_cache.pkl", "wb"))
    print(f"Alignment done in {time.time() - t0:.0f}s, saved alignment_cache.pkl")

al = pd.DataFrame([{"stem": s, "key": str(v[0]), "how": v[2], "shift": v[3], "score": v[4]} for s, v in AL.items()])
al.to_csv(f"{OUTD}/frame_alignment.csv", index=False)
al["fail"] = al["how"] == "fail"
print(f"Mot {len(al)} frame | align hoyeche {100 - al['fail'].mean() * 100:.1f}%")
bad = al.groupby("key")["fail"].mean()
print("Jei camera folder e alignment fail > 5%:")
print((bad[bad > 0.05] * 100).round(1).to_string())

ZCACHE = {}
def zone_polys(stem):
    if stem in ZCACHE:
        return ZCACHE[stem]
    v = AL.get(stem)
    res = None
    if v is not None and v[1] is not None:
        res = []
        for poly in ZONES[v[0]]:
            pts = np.float32([[x * W0, y * H0] for x, y in poly]).reshape(-1, 1, 2)
            res.append(Path(cv2.perspectiveTransform(pts, v[1]).reshape(-1, 2) / [W0, H0]))
    ZCACHE[stem] = res
    return res

In [ ]:
# CELL 4: split, ground truth, common frame list, metric helper
def list_split(s):
    d = f"{SH}/{s}/images"
    return sorted(os.path.join(d, f) for f in os.listdir(d) if f.lower().endswith(EXTS))

def boxes_from_label(lp):
    out = []
    if os.path.exists(lp):
        with open(lp) as fh:
            for line in fh:
                q = line.split()
                if len(q) >= 5:
                    c, xc, yc, w, h = int(q[0]), *map(float, q[1:5])
                    out.append((c, [xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2]))
    return out

PATHS = {s: list_split(s) for s in ("train", "val", "test")}
GT = {s: {stem_of(p): boxes_from_label(f"{SH}/{s}/labels/{stem_of(p)}.txt") for p in PATHS[s]} for s in ("val", "test")}
print({s: len(v) for s, v in PATHS.items()})

def foot(b):
    x0, y0, x1, y1 = b
    return (min(max((x0 + x1) / 2, 0.001), 0.999), min(max(y1, 0.001), 0.995))

def any_in_zone(boxes, stem):
    z = zone_polys(stem)
    if z is None:
        return False
    return any(any(P.contains_point(foot(b)) for P in z) for c, b in boxes if c not in PED)

def enc_pred(dets, stem, thr):
    return any_in_zone([(c, b) for c, b, s in dets if s >= thr], stem)

def enc_frames(split):
    return sorted(s for s in GT[split] if key_of_stem.get(s) in ZONES and zone_polys(s) is not None)

ENC = {s: enc_frames(s) for s in ("val", "test")}
GT_ENC = {s: {st: any_in_zone(GT[s][st], st) for st in ENC[s]} for s in ("val", "test")}
print(f"Encroachment frame (Location 1/3/4, zone align hoyeche): val {len(ENC['val'])} | test {len(ENC['test'])}")
print(f"Test e annotation onujayi Encroached: {np.mean(list(GT_ENC['test'].values())) * 100:.1f}%")

def summarize(p, g):
    p, g = np.asarray(p, dtype=bool), np.asarray(g, dtype=bool)
    n = len(p)
    tp, fp = int((p & g).sum()), int((p & ~g).sum())
    fn, tn = int((~p & g).sum()), int((~p & ~g).sum())
    po, pm, pg = (tp + tn) / n, (tp + fp) / n, (tp + fn) / n
    pc = pm * pg + (1 - pm) * (1 - pg)
    pr, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {"n": n, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "model_enc_%": pm * 100, "gt_enc_%": pg * 100, "agree_%": po * 100,
            "kappa": (po - pc) / (1 - pc) if pc < 1 else float("nan"),
            "recall_%": rc * 100, "precision_%": pr * 100,
            "f1_%": 2 * pr * rc / max(pr + rc, 1e-9) * 100}

# Congestion: shudhu gari gona (Pedestrian baad); threshold TRAIN annotation theke
def n_vehicle(boxes):
    return sum(1 for b in boxes if b[0] not in PED)

train_counts = np.array([n_vehicle(boxes_from_label(f"{SH}/train/labels/{stem_of(p)}.txt")) for p in PATHS["train"]])
Q33, Q66 = [float(x) for x in np.quantile(train_counts, [0.33, 0.66])]
LEVELS = ["Low", "Medium", "High"]
def level(c):
    return 0 if c <= Q33 else (1 if c <= Q66 else 2)
GT_CNT = {s: {st: n_vehicle(b) for st, b in GT[s].items()} for s in ("val", "test")}
print(f"\nCongestion threshold (train annotation, gari only): Low <= {Q33:g} | Medium <= {Q66:g} | High > {Q66:g}")
d = pd.Series([LEVELS[level(c)] for c in GT_CNT["test"].values()]).value_counts().reindex(LEVELS, fill_value=0)
print("Test set annotation level:", d.to_dict())

In [ ]:
# CELL 5: model loader. Sob model same format e dey: (stem, W, H, [(class, [x1,y1,x2,y2] 0-1, score), ...])
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

ALIAS = {"bike": "Motorcycle", "motorcycle": "Motorcycle", "cycle": "Bicycle", "bicycle": "Bicycle",
         "people": "Pedestrian", "pedestrian": "Pedestrian", "cng": "CNG", "car": "Car", "bus": "Bus",
         "truck": "Truck", "rickshaw": "Rickshaw", "mini-truck": "Mini-Truck"}

def make_ultra(path, kind):
    from ultralytics import YOLO, RTDETR
    m = RTDETR(path) if kind == "rtdetr" else YOLO(path)
    cmap = {i: CLASSES.index(ALIAS[n.strip().lower()]) for i, n in m.names.items()}
    print("  classes:", m.names)
    def predict(paths, conf):
        src = os.path.dirname(paths[0])   # puro folder dao; list dile Ultralytics sob chobi EK batch e ney -> GPU OutOfMemory
        for r in m.predict(source=src, conf=conf, imgsz=640, max_det=300, stream=True, verbose=False):
            H, W = r.orig_shape
            if r.boxes is None or len(r.boxes) == 0:
                yield stem_of(r.path), W, H, []
                continue
            c = [cmap[int(x)] for x in r.boxes.cls.tolist()]
            yield stem_of(r.path), W, H, list(zip(c, r.boxes.xyxyn.tolist(), r.boxes.conf.tolist()))
    return predict, sum(p.numel() for p in m.model.parameters()), m

def make_frcnn(path):
    import torchvision.transforms.functional as TF
    from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    m = fasterrcnn_resnet50_fpn_v2(weights=None, weights_backbone=None)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, 10)
    ck = torch.load(path, map_location="cpu", weights_only=False)
    m.load_state_dict(ck["model_state_dict"] if "model_state_dict" in ck else ck)
    m.roi_heads.detections_per_img = 300
    m.to(DEV).eval()
    @torch.no_grad()
    def predict(paths, conf):
        m.roi_heads.score_thresh = min(conf, 0.05)
        for batch in chunks(paths, 8):
            ims = [Image.open(p).convert("RGB") for p in batch]
            outs = m([TF.to_tensor(im).to(DEV) for im in ims])
            for p, im, o in zip(batch, ims, outs):
                W, H = im.size
                res = []
                for lab, box, sc in zip(o["labels"].tolist(), o["boxes"].tolist(), o["scores"].tolist()):
                    if sc < conf or lab == 0:
                        continue
                    x1, y1, x2, y2 = box
                    res.append((lab - 1, [x1 / W, y1 / H, x2 / W, y2 / H], sc))
                yield stem_of(p), W, H, res
    return predict, sum(p.numel() for p in m.parameters()), m

def make_effdet(path):
    import effdet
    from effdet import get_efficientdet_config, EfficientDet
    sd = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(sd, dict) and "model_state_dict" in sd:
        sd = sd["model_state_dict"]
    cfg = get_efficientdet_config("tf_efficientdet_d3")
    cfg.num_classes = 9
    cfg.image_size = (EFF_IMG, EFF_IMG)
    try:
        cfg.max_det_per_image = 300
    except Exception:
        pass
    net = EfficientDet(cfg, pretrained_backbone=False)
    net.load_state_dict(sd, strict=True)
    Bench = getattr(effdet, "DetBenchPredict", None) or getattr(effdet, "DetBenchEval")
    try:
        bench = Bench(net, cfg)
    except TypeError:
        bench = Bench(net)
    bench.to(DEV).eval()
    MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    raw_cls = []
    @torch.no_grad()
    def predict(paths, conf):
        for p in paths:
            img = Image.open(p).convert("RGB")
            W, H = img.size
            arr = (np.asarray(img.resize((EFF_IMG, EFF_IMG), Image.BILINEAR), dtype=np.float32) / 255.0 - MEAN) / STD
            o = bench(torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(DEV))
            if isinstance(o, (list, tuple)):
                o = o[0]
            if o.dim() == 3:
                o = o[0]
            res = []
            for x1, y1, x2, y2, sc, cl in o.tolist():
                if sc < conf:
                    continue
                if len(raw_cls) < 3000 and sc >= 0.5:
                    raw_cls.append(int(cl))
                c = int(cl) + EFF_SHIFT
                if not (0 <= c <= 8):
                    continue
                res.append((c, [x1 / EFF_IMG, y1 / EFF_IMG, x2 / EFF_IMG, y2 / EFF_IMG], sc))
            yield stem_of(p), W, H, res
    predict.raw_cls = raw_cls
    return predict, sum(p.numel() for p in net.parameters()), bench

def build(name):
    p = F[name]
    if name in ("YOLOv8n", "YOLO11n"):
        return make_ultra(p, "yolo")
    if name == "RT-DETR-L":
        return make_ultra(p, "rtdetr")
    if name == "Faster R-CNN":
        return make_frcnn(p)
    if name == "EfficientDet-D3":
        return make_effdet(p)
    raise ValueError(name)

def safe(name):
    return re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
print("Model loader ready")

In [ ]:
# CELL 6: prottek model -> test set e detection + mAP, validation set e detection (cache kore rakhi)
# Somoy: YOLO ~5 min, RT-DETR ~15 min, Faster R-CNN ~20 min, EfficientDet ~30 min
from torchmetrics.detection import MeanAveragePrecision

def to_px(boxes, W, H):
    return torch.tensor([[b[0] * W, b[1] * H, b[2] * W, b[3] * H] for b in boxes], dtype=torch.float32).reshape(-1, 4)

DETS, MAP_ROWS = {}, []
for name in MODELS_TO_RUN:
    cache = f"{WK}/dets_{safe(name)}.pkl"
    if os.path.exists(cache):
        DETS[name], row = pickle.load(open(cache, "rb"))
        MAP_ROWS.append(row)
        print(f"[{name}] cache theke load holo")
        continue
    print(f"\n[{name}] loading {F[name]}")
    predict, nparam, obj = build(name)
    m_all = MeanAveragePrecision(iou_type="bbox", max_detection_thresholds=[1, 10, 300])
    m_50 = MeanAveragePrecision(iou_type="bbox", iou_thresholds=[0.5], class_metrics=True,
                                max_detection_thresholds=[1, 10, 300])
    test_d, t0 = {}, time.time()
    for i, (stem, W, H, dets) in enumerate(predict(PATHS["test"], 0.001)):
        g = GT["test"][stem]
        pr = [{"boxes": to_px([b for c, b, s in dets], W, H),
               "scores": torch.tensor([s for c, b, s in dets], dtype=torch.float32),
               "labels": torch.tensor([c for c, b, s in dets], dtype=torch.int64)}]
        tg = [{"boxes": to_px([b for c, b in g], W, H),
               "labels": torch.tensor([c for c, b in g], dtype=torch.int64)}]
        m_all.update(pr, tg)
        m_50.update(pr, tg)
        test_d[stem] = [(c, b, s) for c, b, s in dets if s >= 0.10]
        if (i + 1) % 500 == 0:
            print(f"  test {i + 1}/{len(PATHS['test'])} ({time.time() - t0:.0f}s)")
    sec_img = (time.time() - t0) / len(PATHS["test"])
    t1 = time.time()
    val_d = {stem: dets for stem, W, H, dets in predict(PATHS["val"], 0.10)}
    print(f"  val done ({time.time() - t1:.0f}s)")
    if name == "EfficientDet-D3":
        print("  EfficientDet raw class (score>=0.5) histogram:", pd.Series(predict.raw_cls).value_counts().sort_index().to_dict(),
              "-> 1..9 hole EFF_SHIFT=-1 thik")
    ra, r5 = m_all.compute(), m_50.compute()
    row = {"model": name, "params_M": round(nparam / 1e6, 2), "sec_per_img_test": round(sec_img, 4),
           "mAP50": float(ra["map_50"]), "mAP50-95": float(ra["map"]),
           "precision@.5_note": "see Ultralytics check for P/R"}
    cls_ids = r5["classes"].tolist() if r5["classes"].dim() else [int(r5["classes"])]
    ap = r5["map_per_class"].tolist() if r5["map_per_class"].dim() else [float(r5["map_per_class"])]
    for c, v in zip(cls_ids, ap):
        row[f"AP50_{CLASSES[int(c)]}"] = v
    DETS[name] = {"test": test_d, "val": val_d}
    MAP_ROWS.append(row)
    pickle.dump((DETS[name], row), open(cache, "wb"))
    print(f"  [{name}] mAP50 = {row['mAP50']:.4f} | mAP50-95 = {row['mAP50-95']:.4f} | {sec_img * 1000:.0f} ms/img")
    del predict, obj, m_all, m_50
    gc.collect()
    torch.cuda.empty_cache()

map_df = pd.DataFrame(MAP_ROWS).drop(columns=["precision@.5_note"])
map_df.to_csv(f"{OUTD}/detection_map_test.csv", index=False)
print("\n" + "=" * 70 + "\nDETECTION (test set, torchmetrics COCO mAP)\n" + "=" * 70)
print(map_df.round(4).to_string(index=False))

In [ ]:
# CELL 7 (cross-check): YOLO / RT-DETR er Ultralytics official test mAP + Precision/Recall
if RUN_ULTRA_CHECK:
    import yaml
    from ultralytics import YOLO, RTDETR
    yp = f"{WK}/data_eval.yaml"
    yaml.safe_dump({"train": f"{SH}/train/images", "val": f"{SH}/val/images", "test": f"{SH}/test/images",
                    "nc": 9, "names": CLASSES}, open(yp, "w"))
    rows = []
    for name in [m for m in MODELS_TO_RUN if m in ("YOLOv8n", "YOLO11n", "RT-DETR-L")]:
        m = RTDETR(F[name]) if name == "RT-DETR-L" else YOLO(F[name])
        r = m.val(data=yp, split="test", imgsz=640, batch=16, plots=False, verbose=False,
                  project=f"{WK}/ultra_val", name=safe(name))
        rows.append({"model": name, "ultra_mAP50": r.box.map50, "ultra_mAP50-95": r.box.map,
                     "ultra_precision": r.box.mp, "ultra_recall": r.box.mr})
        del m
        torch.cuda.empty_cache()
    ultra_df = pd.DataFrame(rows)
    ultra_df.to_csv(f"{OUTD}/detection_map_test_ultralytics.csv", index=False)
    print(ultra_df.round(4).to_string(index=False))

In [ ]:
# CELL 8: ROADSIDE ENCROACHMENT (validation e threshold bachai -> test e ekbar)
enc_rows, sweeps = [], {}
test_tab = pd.DataFrame({"stem": ENC["test"]})
test_tab["loc"] = [key_of_stem[s][0] for s in ENC["test"]]
test_tab["night"] = [key_of_stem[s][2] == 5 for s in ENC["test"]]
test_tab["gt"] = [GT_ENC["test"][s] for s in ENC["test"]]
gv = [GT_ENC["val"][s] for s in ENC["val"]]

for name in MODELS_TO_RUN:
    dv, dt = DETS[name]["val"], DETS[name]["test"]
    sw = pd.DataFrame([{"thr": t, **summarize([enc_pred(dv.get(s, []), s, t) for s in ENC["val"]], gv)} for t in THRS])
    sw.to_csv(f"{OUTD}/enc_val_sweep_{safe(name)}.csv", index=False)
    sweeps[name] = sw
    thr = float(sw.loc[sw["kappa"].idxmax(), "thr"])
    test_tab[name] = [enc_pred(dt.get(s, []), s, thr) for s in ENC["test"]]
    R = summarize(test_tab[name], test_tab["gt"])
    row = {"model": name, "val_best_thr": thr, **R}
    for L in (1, 3, 4):
        s = test_tab[test_tab["loc"] == L]
        row[f"kappa_{LOC_NAME[L]}"] = summarize(s[name], s["gt"])["kappa"]
    row["kappa_day"] = summarize(test_tab.loc[~test_tab["night"], name], test_tab.loc[~test_tab["night"], "gt"])["kappa"]
    row["kappa_night"] = summarize(test_tab.loc[test_tab["night"], name], test_tab.loc[test_tab["night"], "gt"])["kappa"]
    enc_rows.append(row)

enc_df = pd.DataFrame(enc_rows)
enc_df.to_csv(f"{OUTD}/encroachment_test.csv", index=False)
test_tab.to_csv(f"{OUTD}/encroachment_test_per_frame.csv", index=False)
print("=" * 70 + f"\nENCROACHMENT | test {len(test_tab)} frame (sob model same frame) | "
      f"annotation Encroached {test_tab['gt'].mean() * 100:.1f}%\n" + "=" * 70)
cols = ["model", "val_best_thr", "agree_%", "kappa", "f1_%", "recall_%", "precision_%", "model_enc_%",
        "kappa_Arambag", "kappa_Abul Hotel", "kappa_Bashabo", "kappa_day", "kappa_night"]
print(enc_df[cols].round(3).to_string(index=False))
print("\nPrevalence (annotation Encroached %) location onujayi:",
      (test_tab.groupby("loc")["gt"].mean() * 100).round(1).rename(LOC_NAME).to_dict())

fig, ax = plt.subplots(figsize=(8, 4.5))
for name, sw in sweeps.items():
    ax.plot(sw["thr"], sw["kappa"], marker="o", label=name)
ax.set_xlabel("Confidence threshold")
ax.set_ylabel("Cohen's kappa (validation)")
ax.set_title("Encroachment: validation threshold sweep")
ax.legend()
plt.savefig(f"{OUTD}/enc_val_sweep_all.png", dpi=110, bbox_inches="tight")
plt.show()

In [ ]:
# CELL 9: TRAFFIC CONGESTION (train annotation threshold; model confidence validation e bachai -> test e ekbar)
from sklearn.metrics import f1_score, cohen_kappa_score, confusion_matrix

def count_dets(dets, thr):
    return sum(1 for c, b, s in dets if s >= thr and c not in PED)

val_stems, test_stems = sorted(GT_CNT["val"]), sorted(GT_CNT["test"])
gv_cnt = np.array([GT_CNT["val"][s] for s in val_stems])
gt_cnt = np.array([GT_CNT["test"][s] for s in test_stems])
gv_lvl = np.array([level(c) for c in gv_cnt])
gt_lvl = np.array([level(c) for c in gt_cnt])

cong_tab = pd.DataFrame({"stem": test_stems, "gt_count": gt_cnt, "gt_level": [LEVELS[x] for x in gt_lvl]})
cong_tab["loc"] = [key_of_stem.get(s, (0,))[0] if key_of_stem.get(s) else 0 for s in test_stems]
cong_rows, cms = [], {}
for name in MODELS_TO_RUN:
    dv, dt = DETS[name]["val"], DETS[name]["test"]
    sweep = []
    for t in THRS:
        pc = np.array([count_dets(dv.get(s, []), t) for s in val_stems])
        sweep.append({"thr": t, "macro_f1": f1_score(gv_lvl, [level(c) for c in pc], average="macro"),
                      "mae": np.abs(pc - gv_cnt).mean()})
    sw = pd.DataFrame(sweep)
    sw.to_csv(f"{OUTD}/cong_val_sweep_{safe(name)}.csv", index=False)
    thr = float(sw.loc[sw["macro_f1"].idxmax(), "thr"])
    pc = np.array([count_dets(dt.get(s, []), thr) for s in test_stems])
    pl = np.array([level(c) for c in pc])
    cong_tab[f"{name}_count"] = pc
    cong_tab[f"{name}_level"] = [LEVELS[x] for x in pl]
    cms[name] = confusion_matrix(gt_lvl, pl, labels=[0, 1, 2])
    cong_rows.append({"model": name, "val_best_thr": thr, "n": len(pl),
                      "accuracy_%": (pl == gt_lvl).mean() * 100,
                      "macro_f1_%": f1_score(gt_lvl, pl, average="macro") * 100,
                      "kappa_weighted": cohen_kappa_score(gt_lvl, pl, weights="linear"),
                      "count_MAE": np.abs(pc - gt_cnt).mean(), "count_bias": (pc - gt_cnt).mean(),
                      "far_miss_(Low<->High)": int(((pl == 0) & (gt_lvl == 2)).sum() + ((pl == 2) & (gt_lvl == 0)).sum())})

cong_df = pd.DataFrame(cong_rows)
cong_df.to_csv(f"{OUTD}/congestion_test.csv", index=False)
cong_tab.to_csv(f"{OUTD}/congestion_test_per_image.csv", index=False)
print("=" * 70 + f"\nCONGESTION | test {len(test_stems)} image (sob location) | "
      f"Low <= {Q33:g} < Medium <= {Q66:g} < High (gari only)\n" + "=" * 70)
print(cong_df.round(3).to_string(index=False))
for name, cm in cms.items():
    print(f"\n{name} confusion (row = annotation, column = model):")
    print(pd.DataFrame(cm, index=LEVELS, columns=LEVELS).to_string())

In [ ]:
# CELL 10: McNemar (encroachment, same test frame) + MASTER TABLE
from scipy.stats import binomtest
mc = []
names = list(MODELS_TO_RUN)
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a = test_tab[names[i]] == test_tab["gt"]
        b = test_tab[names[j]] == test_tab["gt"]
        n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
        p = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
        mc.append({"model_A": names[i], "model_B": names[j], "only_A_correct": n01, "only_B_correct": n10,
                   "p_value": p, "significant_(p<0.05)": p < 0.05})
mc_df = pd.DataFrame(mc)
mc_df.to_csv(f"{OUTD}/encroachment_mcnemar.csv", index=False)
print("McNemar (encroachment):")
print(mc_df.round(4).to_string(index=False))

master = (map_df[["model", "params_M", "sec_per_img_test", "mAP50", "mAP50-95"]]
          .merge(enc_df[["model", "val_best_thr", "agree_%", "kappa", "f1_%"]]
                 .rename(columns={"val_best_thr": "enc_thr", "agree_%": "enc_agree_%", "kappa": "enc_kappa", "f1_%": "enc_f1_%"}), on="model")
          .merge(cong_df[["model", "val_best_thr", "accuracy_%", "macro_f1_%", "count_MAE"]]
                 .rename(columns={"val_best_thr": "cong_thr", "accuracy_%": "cong_acc_%", "macro_f1_%": "cong_macro_f1_%"}), on="model"))
if RUN_ULTRA_CHECK:
    master = master.merge(ultra_df[["model", "ultra_mAP50", "ultra_mAP50-95"]], on="model", how="left")
master.to_csv(f"{OUTD}/MASTER_RESULTS.csv", index=False)
print("\n" + "=" * 90 + "\nMASTER RESULTS (70/20/10 split, test set)\n" + "=" * 90)
print(master.round(3).to_string(index=False))

In [ ]:
# CELL 11: sob result ek zip e (download kore Claude ke dao)
import zipfile
zp = f"{WK}/eval_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUTD)):
        z.write(os.path.join(OUTD, f), f)
print("Saved:", zp, f"({os.path.getsize(zp) / 1e6:.1f} MB)")
print("alignment_cache.pkl ar dets_*.pkl /kaggle/working e ache (porer notebook e reuse kora jabe)")